In [1]:
import sqlite3
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.model_selection import train_test_split


/opt/miniconda3/envs/database_clean/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
conn = sqlite3.connect("/Users/reubensantoso/Xu_Lab_Files/c3s_neural_network/C3S.db")
master_df = pd.read_sql_query("SELECT * FROM master", conn)
mqn_df = pd.read_sql_query("SELECT * FROM mqns", conn)
fngr_df = pd.read_sql_query("SELECT * FROM fingerprints", conn)
conn.close()

master_df = master_df.dropna()
print(master_df.shape)
print(mqn_df.shape)
print(fngr_df.shape)

(15187, 12)
(15187, 43)
(15187, 1025)


In [3]:
master_df.head(2)

,g_id,name,adduct,mass,z,mz,ccs,smi,chem_class_label,src_tag,ccs_type,ccs_method
0,CCSBASE_C4B6CF0FE6,1-Methylnicotinamide,[M]+,137.0715,1,137.0715,126.4,C[N+]1=CC=CC(=C1)C(=O)N,small molecule,zhou1016,DT,"single field, calibrated with Agilent tune mix..."
1,CCSBASE_D0DE2590C2,7-Methylguanosine,[M]+,298.1151,1,298.1151,166.5,CN1C=[N+](C2=C1C(=O)N=C(N2)N)[C@H]3[C@@H]([C@@...,small molecule,zhou1016,DT,"single field, calibrated with Agilent tune mix..."


# Building Features

In [4]:
from sklearn.preprocessing import OneHotEncoder

In [5]:
c3s_df = pd.DataFrame()

# adding mz into c3s
c3s_df = master_df[["g_id", "mz", "smi", "ccs"]].copy()

# adding one hot encoded adducts into c3s
adduct_groups = [
    "[M+H]+", "[M+Na]+", "[M-H]-", "[M+NH4]+", "[M+K]+",
    "[M+H-H2O]+", "[M+HCOO]-", "[M+CH3COO]-", "[M+Na-2H]-"
]

adduct_encoding = OneHotEncoder(
    sparse_output=False,
    categories='auto', 
    handle_unknown="infrequent_if_exist",
    min_frequency=30 #needs to be above 5 data samples to be encoded. if below gets grouped to others
)

adduct_encoded = adduct_encoding.fit_transform(master_df[['adduct']])
adduct_encoded_df = pd.DataFrame(
    adduct_encoded, 
    columns=adduct_encoding.get_feature_names_out(['adduct'])
)
adduct_encoded_df["g_id"] = master_df["g_id"].values

# Merge encoded adducts by g_id
c3s_df = c3s_df.merge(adduct_encoded_df, on="g_id", how="left")

# Add MQN features by merging on g_id
c3s_df = c3s_df.merge(mqn_df[["g_id", "adv", "hac", "hbam", "c", "asb"]], on="g_id", how="left")

c3s_df = c3s_df.merge(fngr_df, on="g_id", how="left")

# Final result
print(c3s_df.shape)
c3s_df.head(2)

(15187, 1047)


,g_id,mz,smi,ccs,adduct_[2M-H]-,adduct_[M+2Na-H]+,adduct_[M+CH3COO]-,adduct_[M+Cl]-,adduct_[M+H-H2O]+,adduct_[M+HCOO]-,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,CCSBASE_C4B6CF0FE6,137.0715,C[N+]1=CC=CC(=C1)C(=O)N,126.4,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,CCSBASE_D0DE2590C2,298.1151,CN1C=[N+](C2=C1C(=O)N=C(N2)N)[C@H]3[C@@H]([C@@...,166.5,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,0


In [6]:
use_subset = False

if use_subset:
    c3s_df = c3s_df.sample(n=1000, random_state=42).reset_index(drop=True)
    fngr_df = fngr_df[fngr_df["g_id"].isin(c3s_df["g_id"])]

print(c3s_df.shape)
print(fngr_df.shape)

(15187, 1047)
(15187, 1025)


In [7]:
# ensure g_id is a *column*, then set it once
if c3s_df.index.name == "g_id":
    c3s_df = c3s_df.reset_index()      # puts g_id back into columns
c3s_df = c3s_df.set_index("g_id")


y = torch.tensor(c3s_df["ccs"].values, dtype=torch.float32)

# drop target ➜ expand any residual categoricals ➜ numeric only
feature_df = c3s_df.drop(columns=["ccs"])

numeric_df  = feature_df.select_dtypes(include=[np.number])   # ints + floats only
#takes out g_id, smi, and ccs

X = torch.tensor(numeric_df.values, dtype=torch.float32)

print("entries dim:", X.shape[0])
print("feature dim:", X.shape[1])

entries dim: 15187
feature dim: 1044


# building edge index

In [8]:
SIM_THRESHOLD    = 0.70              # tanimoto cutoff
VAL_FRAC         = 0.10             # 15 % validation
TEST_FRAC        = 0.20             # 15 % test
SEED             = 42

In [9]:
fp_cols  = [c for c in fngr_df.columns if c != "g_id"]
fp_mat   = fngr_df.set_index("g_id").loc[c3s_df.index, fp_cols].values.astype(bool)
num_nodes = fp_mat.shape[0]

edge_li = []
for i, j in itertools.combinations(range(num_nodes), 2):
    inter  = np.logical_and(fp_mat[i], fp_mat[j]).sum()
    union  = np.logical_or(fp_mat[i], fp_mat[j]).sum()
    tanimoto = inter / union if union else 0.0
    if tanimoto >= SIM_THRESHOLD:          # keep only strong similarities
        edge_li.extend([(i, j), (j, i)])   # undirected

edge_index = torch.tensor(edge_li, dtype=torch.long).t().contiguous()
print(f"edges (pairs ≥ {SIM_THRESHOLD}):", edge_index.size(1) // 2)

edges (pairs ≥ 0.7): 1700074


# Split

In [10]:
all_idx = np.arange(num_nodes)
train_idx, tmp_idx = train_test_split(all_idx, test_size=VAL_FRAC+TEST_FRAC,
                                      random_state=SEED, shuffle=True)
rel_test_frac = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
val_idx, test_idx = train_test_split(tmp_idx, test_size=rel_test_frac,
                                     random_state=SEED, shuffle=True)

mask = lambda: torch.zeros(num_nodes, dtype=torch.bool)
train_mask, val_mask, test_mask = mask(), mask(), mask()
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

data = Data(x=X, y=y, edge_index=edge_index,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

# Model

In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [12]:
EPOCHS = 200
PATIENCE = 30

In [13]:
class GAT_NodeLevel(nn.Module):
    """
    Three-layer GAT for CCS regression.
    Layout matches your SAGEConv model:
        GAT → BN → ReLU  ×3  → Linear
    """
    def __init__(self,
                 in_dim:   int,
                 hidden1:  int,
                 hidden2:  int ,
                 hidden3:  int ,
                 heads1:   int ,
                 heads2:   int ,
                 drop:     float):
        super().__init__()
        self.drop = drop

        # Layer 1 (concatenate  heads ⇒ hidden1 * heads1 channels)
        self.gat1 = GATConv(in_dim, hidden1, heads=heads1)
        self.bn1  = nn.BatchNorm1d(hidden1 * heads1)

        # Layer 2 (concatenate again)
        self.gat2 = GATConv(hidden1 * heads1, hidden2, heads=heads2)
        self.bn2  = nn.BatchNorm1d(hidden2 * heads2)

        # Layer 3 (single head, concat=False ⇒ size = hidden3)
        self.gat3 = GATConv(hidden2 * heads2, hidden3,
                            heads=4, concat=False, dropout=drop)
        self.bn3  = nn.BatchNorm1d(hidden3)

        self.out  = nn.Linear(hidden3, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # --- 1 ---
        x = self.gat1(x, edge_index)
        # x = self.bn1(x)
        x = F.relu(x)
        # x = F.dropout(x, p=self.drop, training=self.training)

        # --- 2 ---
        x = self.gat2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.drop, training=self.training)

        # --- 3 ---
        x = self.gat3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        return self.out(x).squeeze(-1)      # [num_nodes]

In [14]:

def rel_err(pred, true):
    """Relative error in %."""
    return torch.mean(100 * torch.abs(pred - true) / true).item()

In [15]:
def train_model(model, data, train_mask, val_mask, lr, weight_decay, 
                l1_lambda, epochs=EPOCHS, patience=PATIENCE, 
                model_path="best_model.pth"):
    """
    Train and evaluate a model with L1 regularization and early stopping.
    Returns the training history.
    """
    criterion = nn.MSELoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Setup scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.1)
    
    # Initialize early stopping
    best_val_mre = float("inf")
    best_train_mre  = float("inf")
    best_val_epoch = -1
    patience_ctr = 0
    history = []
    
    print("🔹 Training with early stopping and L1 regularization …")
    for epoch in range(1, epochs + 1):
        # ---- train ----
        model.train()
        optimizer.zero_grad()
        out = model(data)
        
        # Calculate MSE loss
        mse_loss = criterion(out[train_mask], data.y[train_mask])
        
        # Add L1 regularization without creating a separate computational graph
        l1_loss = 0
        for param in model.parameters():
            l1_loss += torch.sum(torch.abs(param))
        
        # Combined loss
        total_loss = mse_loss + l1_lambda * l1_loss
        
        # Single backward pass
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        # ---- evaluate ----
        model.eval()
        with torch.no_grad():
            val_loss = criterion(out[val_mask], data.y[val_mask]).item()
            tr_mre = rel_err(out[train_mask], data.y[train_mask])
            val_mre = rel_err(out[val_mask], data.y[val_mask])
        
        history.append({
            "epoch": epoch,
            "train_mse": mse_loss.item(),
            "val_mse": val_loss,
            "train_mre": tr_mre,
            "val_mre": val_mre,
            "lr": scheduler.get_last_lr()[0],
        })
        
        print(f"epoch {epoch:4d}  tr_MSE={mse_loss.item():.4f}  tr_MRE={tr_mre:.2f}%  "
              f"val_MRE={val_mre:.2f}%  lr={scheduler.get_last_lr()[0]:.2e}")
        
        # Check for early stopping
        if tr_mre < best_train_mre:
            best_train_mre = tr_mre

        if val_mre < best_val_mre - 1e-6:
            best_val_mre = val_mre
            best_val_epoch = epoch
            patience_ctr = 0
            
            # Save model
            torch.save(model.state_dict(), model_path)
        else:
            patience_ctr += 1
        
        if patience_ctr >= patience:
            print(f"🛑 Early stopping at epoch {epoch}"
                  f" (no val-MRE improvement for {patience} epochs)")
            break
    
    print(f"Best train-MRE: {best_train_mre:.2f}%")
    print(f"Best val-MRE: {best_val_mre:.2f}% at epoch {best_val_epoch}")
    return pd.DataFrame(history)

def plot_training_history(history_df, save_path=None):
    """
    Plot training and validation metrics.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot Mean Relative Error
    ax1.plot(history_df['epoch'], history_df['train_mre'], label='Train MRE')
    ax1.plot(history_df['epoch'], history_df['val_mre'], label='Validation MRE')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Mean Relative Error (%)')
    ax1.set_title('Training and Validation MRE')
    ax1.legend()
    ax1.grid(True)
    
    # Plot Mean Squared Error
    ax2.plot(history_df['epoch'], history_df['train_mse'], label='Train MSE')
    ax2.plot(history_df['epoch'], history_df['val_mse'], label='Validation MSE')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Mean Squared Error')
    ax2.set_title('Training and Validation MSE')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        
    plt.show()

In [16]:
# ========= hyper‑parameters ==========
# Default values that will be overridden by grid search
HEADS1 = 8
HEADS2 = 4
HIDDEN1 = 128
HIDDEN2 = 64
HIDDEN3 = 32
DROPOUT = 0.3

LR = 0.03
WEIGHT_DECAY = 0.07
L1_LAMBDA = 0.001

# Ensure reproducibility
torch.manual_seed(42)

In [ ]:
# Ensure reproducibility
torch.manual_seed(42)

# ========= Grid Search Configuration ==========
# Define the parameter grid to search
# param_grid = {
#     'lr': [0.005, 0.01, 0.02],
#     'weight_decay': [0.05, 0.1, 0.015],
#     'heads1': [4, 8],
#     'heads2': [2, 4],
#     'dropout': [0.2, 0.3],
#     'hidden1': [64, 128],
#     'hidden2': [32, 64],
#     'hidden3': [16, 32]
# }


# Initialize model with best hyperparameters from grid search
model = GAT_NodeLevel(
    in_dim=X.shape[1],
    hidden1=HIDDEN1,
    hidden2=HIDDEN2,
    hidden3=HIDDEN3,
    heads1=HEADS1,
    heads2=HEADS2,
    drop=DROPOUT
)

# Train model
history_df = train_model(
    model, data, train_mask, val_mask,
    lr=LR, 
    weight_decay=WEIGHT_DECAY,
    l1_lambda=L1_LAMBDA
)

# Plot training history
plot_training_history(history_df, save_path="training_history.png")


🔹 Training with early stopping and L1 regularization …
epoch    1  tr_MSE=49494.5430  tr_MRE=100.04%  val_MRE=100.04%  lr=3.00e-02
epoch    2  tr_MSE=49280.9531  tr_MRE=99.88%  val_MRE=99.87%  lr=3.00e-02
epoch    3  tr_MSE=48985.3125  tr_MRE=99.65%  val_MRE=99.65%  lr=3.00e-02
epoch    4  tr_MSE=48753.7539  tr_MRE=99.40%  val_MRE=99.40%  lr=3.00e-02
epoch    5  tr_MSE=48524.6797  tr_MRE=99.14%  val_MRE=99.13%  lr=3.00e-02
epoch    6  tr_MSE=48286.5977  tr_MRE=98.87%  val_MRE=98.87%  lr=2.99e-02
epoch    7  tr_MSE=48033.8750  tr_MRE=98.60%  val_MRE=98.59%  lr=2.99e-02
epoch    8  tr_MSE=47765.2500  tr_MRE=98.31%  val_MRE=98.31%  lr=2.99e-02
epoch    9  tr_MSE=47477.1641  tr_MRE=98.02%  val_MRE=98.02%  lr=2.99e-02
epoch   10  tr_MSE=47170.8633  tr_MRE=97.71%  val_MRE=97.71%  lr=2.98e-02
epoch   11  tr_MSE=46843.2773  tr_MRE=97.38%  val_MRE=97.38%  lr=2.98e-02
epoch   12  tr_MSE=46491.4805  tr_MRE=97.05%  val_MRE=97.04%  lr=2.98e-02
epoch   13  tr_MSE=46115.5195  tr_MRE=96.69%  val_MRE=9

In [ ]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    out = model(data)
    val_mre = rel_err(out[val_mask], data.y[val_mask])
    print(f"Final validation MRE: {val_mre:.2f}%")

In [ ]:
with torch.no_grad():
    diff = torch.abs(out[val_mask] - data.y[val_mask])
    rel  = 100 * diff / data.y[val_mask]
    topk = torch.topk(rel, 5)          # worst 5 %
    print("Worst 5 val relative errors:", rel[topk.indices].cpu().numpy())


In [ ]:
# # ---------- 10. Save per‑molecule results -----
# name_col = "name" if "name" in master_df.columns else None
# names = master_df.set_index("g_id").loc[c3s_df.index, name_col].values if name_col else c3s_df.index.values

# results_df = pd.DataFrame({
#     "molecule": names[test_mask],
#     "ccs_true": data.y[test_mask].cpu().numpy(),
#     "ccs_pred": preds[test_mask].cpu().numpy()})

# results_df.to_csv("test_predictions.csv", index=False)
# print("📄 Saved test predictions → test_predictions.csv")
